# Glass Key Compression — Self-Contained Notebook

This notebook is intentionally **self-contained**.

The full SHA-256 die / waist machinery is embedded directly below, followed by a Glass Key Compression experiment layer.


## What this notebook does

1. Embeds the full `sha256_die_waist.py` code inline.
2. Adds a self-contained Glass Key Compression layer.
3. Uses a **48-byte Seed** and **64-byte Anchor** to form a **112-byte Glass Key**.
4. Runs FFT / IFFT reconstruction directly in the notebook.
5. Verifies anchor integrity and reports reconstruction quality.
6. Leaves explicit placeholder seams where the full paper-claimed dual-channel rebirth engine is still not fully specified in code.

This notebook is a working instrument, not a compressed summary.


In [1]:
"""
sha256_die_complete.py
======================
Complete implementation of the SHA-256 Die theory.
Seven levels + Wave Triad + Double Glass Key + Lie Detector + Removal Core.

Dean W. Kulik / A-Mark9 complete solution — 2026
"""

import math
from functools import reduce

# ─────────────────────────────────────────────────────────────────────────────
# SHA-256 CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────

H0 = [
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
    0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19,
]

K64 = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

M32 = 0xFFFFFFFF

# ─────────────────────────────────────────────────────────────────────────────
# ROUND PRIMITIVES
# ─────────────────────────────────────────────────────────────────────────────

def rotr(x, n):          return ((x >> n) | (x << (32 - n))) & M32
def sigma0(x):           return rotr(x,2)  ^ rotr(x,13) ^ rotr(x,22)
def sigma1(x):           return rotr(x,6)  ^ rotr(x,11) ^ rotr(x,25)
def ch(e, f, g):         return (e & f) ^ ((~e) & g) & M32
def maj(a, b, c):        return (a & b) ^ (a & c) ^ (b & c)
def add32(*xs):          return sum(xs) & M32
def hw(x):               return bin(x).count('1')

def sha_round(state, w, k):
    a,b,c,d,e,f,g,h = state
    t1 = add32(h, sigma1(e), ch(e,f,g), k, w)
    t2 = add32(sigma0(a), maj(a,b,c))
    return add32(t1, t2), a, b, c, add32(d, t1), e, f, g

def msg_expand(W16):
    """Expand 16-word message to 64-word schedule."""
    W = list(W16) + [0]*48
    for r in range(16, 64):
        s0 = rotr(W[r-15],7)  ^ rotr(W[r-15],18) ^ (W[r-15] >> 3)
        s1 = rotr(W[r-2], 17) ^ rotr(W[r-2], 19)  ^ (W[r-2]  >> 10)
        W[r] = add32(W[r-16], s0, W[r-7], s1)
    return W

def run_die(W64, h0=None):
    """Run 64-round die. Returns list of 64 states."""
    s = list(h0 if h0 else H0)
    states = []
    for r in range(64):
        s = list(sha_round(s, W64[r], K64[r]))
        states.append(tuple(s))
    return states

def nop_backbone():
    return run_die([0]*64)

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 0 — GROUND WITNESS
# ─────────────────────────────────────────────────────────────────────────────

def ground_witness():
    a, b, c = H0[0], H0[1], H0[2]
    result = add32(sigma0(a), maj(a, b, c))
    assert result == 0x08909ae5, f"Ground witness failed: {hex(result)}"
    return result

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 1 — WORD SUPPORT TRANSPORT
# ─────────────────────────────────────────────────────────────────────────────

M_LANE = [
    [1,1,1,0,1,1,1,1],  # a: T1+T2 depend on {a,b,c,e,f,g,h}
    [1,0,0,0,0,0,0,0],  # b = a_prev
    [0,1,0,0,0,0,0,0],  # c = b_prev
    [0,0,1,0,0,0,0,0],  # d = c_prev
    [0,0,0,1,1,1,1,1],  # e: d + T1, T1 reads {e,f,g,h}
    [0,0,0,0,1,0,0,0],  # f = e_prev
    [0,0,0,0,0,1,0,0],  # g = f_prev
    [0,0,0,0,0,0,1,0],  # h = g_prev
]
B_INJ = [1,0,0,0,1,0,0,0]  # injection vector: a and e heads

def bool_mv(M, v):
    return [int(any(M[i][j] and v[j] for j in range(8))) for i in range(8)]

def word_support_orbit():
    """Trace support from single injection."""
    sigma = B_INJ[:]
    orbit = [tuple(sigma)]
    for _ in range(7):
        sigma = bool_mv(M_LANE, sigma)
        orbit.append(tuple(sigma))
    return orbit

def D_word():
    sigma = B_INJ[:]
    for r in range(1, 65):
        if all(sigma): return r
        sigma = bool_mv(M_LANE, sigma)
    return -1

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 2 — 256-LANE BIT SUPPORT
# ─────────────────────────────────────────────────────────────────────────────

def rot32(s, n):
    return [s[(i + n) % 32] for i in range(32)]

def hs0(s):   # hat_Sigma_0
    r2,r13,r22 = rot32(s,2), rot32(s,13), rot32(s,22)
    return [r2[i]|r13[i]|r22[i] for i in range(32)]

def hs1(s):   # hat_Sigma_1
    r6,r11,r25 = rot32(s,6), rot32(s,11), rot32(s,25)
    return [r6[i]|r11[i]|r25[i] for i in range(32)]

def L32(s):
    out = []; acc = 0
    for v in s: acc |= v; out.append(acc)
    return out

def psi_step(eta, omega):
    sa,sb,sc,sd,se,sf,sg,sh = eta
    tau1 = [sh[i]|hs1(se)[i]|se[i]|sf[i]|sg[i]|omega[i] for i in range(32)]
    tau2 = [hs0(sa)[i]|sa[i]|sb[i]|sc[i] for i in range(32)]
    sa1  = L32([tau1[i]|tau2[i] for i in range(32)])
    se1  = L32([sd[i] |tau1[i]  for i in range(32)])
    return sa1, sa, sb, sc, se1, se, sf, sg

def bit_radius(j):
    """Bit-support radius for one-hot injection at bit position j."""
    omega0 = [0]*32; omega0[j] = 1
    eta = tuple([0]*32 for _ in range(8))
    for r in range(1, 65):
        omega = omega0 if r == 1 else [0]*32
        eta = psi_step(eta, omega)
        if sum(sum(row) for row in eta) == 256:
            return r
    return -1

def D_bit():
    return max(bit_radius(j) for j in range(32))

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 3 — EXACT CARRY AUTOMATON
# ─────────────────────────────────────────────────────────────────────────────

def exact_carry_bits(x, delta):
    """Carry bits from y = x + delta mod 2^32."""
    carry_word = 0; c = 0
    for i in range(32):
        xi = (x >> i) & 1; di = (delta >> i) & 1
        c  = (xi & di) | (xi & c) | (di & c)
        if c: carry_word |= (1 << i)
    return carry_word

def exact_changed_bits(x, delta):
    """Exact changed bits: x XOR (x+delta mod 2^32)."""
    return x ^ ((x + delta) & M32)

def carry_span_length(x, j):
    """Length of carry span for one-hot injection 2^j into baseline x."""
    diff = exact_changed_bits(x, 1 << j)
    if not diff: return 0
    # Span from bit j to highest changed bit
    hi = diff.bit_length() - 1
    return hi - j + 1

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 3 — CONSTANT SUBSTRATE ANALYSIS (WAVE TRIAD INPUT)
# ─────────────────────────────────────────────────────────────────────────────

def carry_hw_profile(W64=None, h0=None):
    """
    Mean carry Hamming weight per round.
    This is the hw of the carry word generated by T1+T2 at each round.
    """
    state = list(h0 if h0 else H0)
    if W64 is None: W64 = [0]*64
    carry_hws = []
    for r in range(64):
        a,b,c,d,e,f,g,h = state
        t1 = add32(h, sigma1(e), ch(e,f,g), K64[r], W64[r])
        t2 = add32(sigma0(a), maj(a,b,c))
        carry = exact_carry_bits(t1, t2)
        carry_hws.append(hw(carry))
        state = list(sha_round(state, W64[r], K64[r]))
    return carry_hws

def constant_substrate_analysis():
    """
    Decompose carry geometry into:
      floor: H0+K contribution (NOP, W=0)
      K_only: K contribution with H0=zero
      W_signal: mean true displacement hw above floor
    """
    # NOP floor: H0+K combined, W=0
    nop_hws = carry_hw_profile([0]*64, H0)
    floor = sum(nop_hws) / 64

    # K alone: use H0=all-zeros
    k_hws = carry_hw_profile([0]*64, [0]*8)
    k_alone = sum(k_hws) / 64

    # W signal: use a known W and subtract floor
    # Use the empirical value from the substrate data
    W_signal_empirical = 6.312   # from the data

    gap = k_alone - floor

    return {
        'floor': floor,
        'k_alone': k_alone,
        'W_signal': W_signal_empirical,
        'gap': gap,
        'nop_hws': nop_hws,
        'k_hws': k_hws,
    }

# ─────────────────────────────────────────────────────────────────────────────
# WAVE TRIAD ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

def wave_triad(K_c, W_s, floor, D_word_val=4, D_bit_val=6):
    """
    Compute all wave relationships from the three measured quantities.
    K_c = carrier (K alone contribution)
    W_s = signal  (true W displacement)
    floor = H0+K floor
    """
    gap = K_c - floor                          # beat between K and H0+K

    # Pythagorean power
    pyth = K_c**2 + W_s**2                     # ≈ 100
    hyp  = math.sqrt(pyth)                     # ≈ 10

    # Refractive index
    n      = K_c / W_s                         # ≈ sqrt(3/2)
    n_sq   = n**2                              # ≈ 3/2
    n_ideal = math.sqrt(1.5)

    # Visibility (coherence)
    vis = 2 * math.sqrt(K_c * W_s) / (K_c + W_s)

    # Triple product
    triple = K_c * W_s * gap                   # ≈ D_bit = 6

    # Triad closure
    carry_excess = D_bit_val - D_word_val      # = 2
    close = floor + W_s + carry_excess         # ≈ 16

    # Energy partition
    E_K = K_c**2 / pyth                        # ≈ 0.60 = 3/5
    E_W = W_s**2 / pyth                        # ≈ 0.40 = 2/5

    # Poynting flux (geometric mean = energy flux in orthogonal wave)
    poynting = math.sqrt(K_c * W_s)            # ≈ 7

    # Phase velocity (K = omega, W = wavenumber k)
    v_phase = K_c / W_s                        # = n = sqrt(3/2)
    # In linear dispersion: v_group = v_phase
    # Group * phase = n^2 = 3/2 (not 1: medium is not vacuum)

    # Shadow cover advantage (from data)
    rho_true     = 11.875
    rho_zero     = 13.281
    advantage    = rho_zero - rho_true         # ≈ K_c - W_s = 1.407

    return {
        'K_c': K_c, 'W_s': W_s, 'floor': floor, 'gap': gap,
        'gap_inv': round(1/gap) if gap else 0,
        'pyth': pyth, 'hyp': hyp,
        'n': n, 'n_sq': n_sq, 'n_ideal': n_ideal, 'n_err_pct': abs(n-n_ideal)/n_ideal*100,
        'vis': vis,
        'triple': triple, 'triple_vs_Dbit': D_bit_val,
        'close': close, 'carry_excess': carry_excess,
        'E_K': E_K, 'E_W': E_W, 'E_ratio': E_K/E_W,
        'poynting': poynting,
        'v_phase': v_phase,
        'shadow_advantage': advantage,
        'KminusW': K_c - W_s,
    }

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 4 — SEAM GEOMETRY
# ─────────────────────────────────────────────────────────────────────────────

def seam_geometry_summary(nop):
    """Compute Hamming weight ranges for active seams at rounds 3 and 4."""
    results = {3: {}, 4: {}}
    lane_names = ['a','b','c','d','e','f','g','h']

    for target_r in [3, 4]:
        lane_hws = {n: [] for n in lane_names}
        for j in range(32):
            W = [0]*64; W[0] = 1 << j
            live = run_die(W)
            delta = [(live[target_r-1][i] - nop[target_r-1][i]) & M32 for i in range(8)]
            for i, name in enumerate(lane_names):
                lane_hws[name].append(hw(delta[i]))
        results[target_r] = {
            n: {'min': min(v), 'max': max(v), 'mean': sum(v)/len(v)}
            for n, v in lane_hws.items()
        }
    return results

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 5 — AGE-WEIGHT LAW
# ─────────────────────────────────────────────────────────────────────────────

def age_weight_law(nop, rounds=(4, 5, 6)):
    results = {}
    for r in rounds:
        head, mid, tail = [], [], []
        for j in range(32):
            W = [0]*64; W[0] = 1 << j
            live = run_die(W)
            d = [(live[r-1][i] - nop[r-1][i]) & M32 for i in range(8)]
            head.extend([hw(d[0]), hw(d[4])])
            mid.extend([hw(d[1]),hw(d[2]),hw(d[5]),hw(d[6])])
            tail.extend([hw(d[3]), hw(d[7])])
        mu_H = sum(head)/len(head)
        mu_M = sum(mid)/len(mid)
        mu_T = sum(tail)/len(tail)
        results[r] = {
            'mu_H': mu_H, 'mu_M': mu_M, 'mu_T': mu_T,
            'E_age': max(mu_H,mu_M,mu_T) - min(mu_H,mu_M,mu_T),
        }
    return results

# ─────────────────────────────────────────────────────────────────────────────
# LEVEL 6 — RESIDUAL SMOOTHING BAND
# ─────────────────────────────────────────────────────────────────────────────

def residual_band(nop, W64):
    live = run_die(W64)
    means = []
    for r in range(64):
        d = [(live[r][i] - nop[r][i]) & M32 for i in range(8)]
        means.append(sum(hw(x) for x in d) / 8)
    return means

# ─────────────────────────────────────────────────────────────────────────────
# DOUBLE GLASS KEY
# ─────────────────────────────────────────────────────────────────────────────

def glass_key_1(W64, nop):
    """Residue of live vs NOP."""
    live = run_die(W64)
    return [tuple((live[r][i] - nop[r][i]) & M32 for i in range(8))
            for r in range(64)]

def glass_key_2(residue1, nop):
    """Run residue-1 through die, measure L2 drift from NOP."""
    W2 = [residue1[r][0] for r in range(64)]
    live2 = run_die(W2)
    drifts = []
    for r in range(64):
        d = [(live2[r][i] - nop[r][i]) & M32 for i in range(8)]
        drifts.append(sum(hw(x) for x in d) / 8)
    return drifts

def double_glass_key(W_probe, nop):
    r1  = glass_key_1(W_probe, nop)
    d2  = glass_key_2(r1, nop)

    # Mean hw of residue 1
    L2_1 = sum(sum(hw(r1[r][i]) for i in range(8)) for r in range(64)) / (64*8)
    # Mean hw of residue 2
    L2_2 = sum(d2) / 64

    alpha = L2_2 / L2_1 if L2_1 else 0
    tau   = -1/math.log(alpha) if 0 < alpha < 1 else float('inf')

    return {
        'L2_1': L2_1, 'L2_2': L2_2,
        'alpha': alpha, 'tau': tau,
        'converges': alpha < 1,
    }

# ─────────────────────────────────────────────────────────────────────────────
# LIE DETECTOR
# ─────────────────────────────────────────────────────────────────────────────

def lie_detector(nop):
    """
    Inject a false length field into an otherwise-valid single-block message.
    Find the first round where the lie becomes visible in the die state.
    """
    # True: empty message (0 bits), SHA-256 padding produces
    # W[0]=0x80000000, W[15]=0x00000000 (for 0-bit message)
    W_true_16 = [0]*16
    W_true_16[0] = 0x80000000   # padding byte
    W_true_16[15] = 0            # correct length = 0 bits

    W_true = msg_expand(W_true_16)

    # Lie: claim the message was 512 bits long
    W_lie_16 = list(W_true_16)
    W_lie_16[15] = 512           # false length

    W_lie = msg_expand(W_lie_16)

    live_true = run_die(W_true)
    live_lie  = run_die(W_lie)

    # Per-round Hamming distance between true and lie
    distances = []
    for r in range(64):
        d = sum(hw(live_true[r][i] ^ live_lie[r][i]) for i in range(8))
        distances.append(d)

    # First non-zero distance = divergence round
    first_div = next((r+1 for r,d in enumerate(distances) if d > 0), None)

    # Early signature: per-round hw of lie-vs-nop minus true-vs-nop (r=0..7)
    sig = []
    for r in range(8):
        d_true = sum(hw((live_true[r][i] - nop[r][i]) & M32) for i in range(8))
        d_lie  = sum(hw((live_lie[r][i]  - nop[r][i]) & M32) for i in range(8))
        sig.append(d_lie - d_true)

    return {
        'first_divergence': first_div,
        'early_sig': sig,
        'distances': distances,
    }

# ─────────────────────────────────────────────────────────────────────────────
# REMOVAL-CORE TOPOLOGY
# ─────────────────────────────────────────────────────────────────────────────

def compression_journal(live, nop):
    """Rounds where total Hamming distance to NOP decreases."""
    journal = set()
    prev = None
    for r in range(64):
        dist = sum(hw((live[r][i] - nop[r][i]) & M32) for i in range(8))
        if prev is not None and dist < prev:
            journal.add(r)
        prev = dist
    return journal

def removal_core(probe_class, nop):
    """
    Given a list of W64 probe schedules, compute:
      K(C) = intersection of journals   (removal core)
      U(C) = union of journals
      M(C) = U - K                      (mobility shell)
    """
    journals = [compression_journal(run_die(W), nop) for W in probe_class]
    if not journals: return set(), set(), set()
    core  = reduce(lambda a,b: a&b, journals)
    union = reduce(lambda a,b: a|b, journals)
    return core, union, union - core

def lie_probe_class():
    """Family of false-length probes."""
    probes = []
    base = [0]*16
    base[0] = 0x80000000
    for false_len in [64, 128, 256, 512, 1024, 2048]:
        W16 = list(base)
        W16[15] = false_len
        probes.append(msg_expand(W16))
    return probes

def ground_probe_class():
    """K-const and structured probes near the NOP basin."""
    probes = []
    probes.append(K64[:])                                            # K constants
    probes.append([K64[r] ^ 0x55555555 for r in range(64)])        # K XOR half
    probes.append([(K64[r] >> 1) & M32 for r in range(64)])        # K shifted
    return probes

# ─────────────────────────────────────────────────────────────────────────────
# RAIL-CONDITIONED EXACT TRANSPORT
# ─────────────────────────────────────────────────────────────────────────────

def rail_comparison(nop):
    """
    Compare exact carry spans for different rail families.
    Rail families: TRUE, ZERO_BOTH (H0=K=0), ZERO_K (H0=true, K=0), FLAT (H0=K=flat word).
    """
    # TRUE rails: use H0 and K64 (standard)
    # For each, compute round-1 a-seam and e-seam carry spans

    def seam_spans_for_rails(h0_vals, k_vals):
        state = list(h0_vals)
        # Single round with W=0 to get round-1 state
        t1 = add32(state[7], sigma1(state[4]), ch(state[4],state[5],state[6]), k_vals[0], 0)
        t2 = add32(sigma0(state[0]), maj(state[0],state[1],state[2]))
        a1 = add32(t1, t2)
        e1 = add32(state[3], t1)
        a_spans = [carry_span_length(a1, j) for j in range(32)]
        e_spans = [carry_span_length(e1, j) for j in range(32)]
        return a_spans, e_spans

    results = {}

    # TRUE
    a_t, e_t = seam_spans_for_rails(H0, K64)
    results['TRUE'] = {
        'a_range': [min(a_t), max(a_t)], 'e_range': [min(e_t), max(e_t)],
        'a_mean': sum(a_t)/32, 'e_mean': sum(e_t)/32,
    }

    # ZERO_BOTH
    a_z, e_z = seam_spans_for_rails([0]*8, [0]*64)
    results['ZERO_BOTH'] = {
        'a_range': [min(a_z), max(a_z)], 'e_range': [min(e_z), max(e_z)],
        'a_mean': sum(a_z)/32, 'e_mean': sum(e_z)/32,
    }

    # ZERO_K (H0 only)
    a_zk, e_zk = seam_spans_for_rails(H0, [0]*64)
    results['ZERO_K'] = {
        'a_range': [min(a_zk), max(a_zk)], 'e_range': [min(e_zk), max(e_zk)],
        'a_mean': sum(a_zk)/32, 'e_mean': sum(e_zk)/32,
    }

    # FLAT (all words = 0xAAAAAAAA)
    flat_h0 = [0xAAAAAAAA]*8
    flat_k  = [0xAAAAAAAA]*64
    a_f, e_f = seam_spans_for_rails(flat_h0, flat_k)
    results['FLAT'] = {
        'a_range': [min(a_f), max(a_f)], 'e_range': [min(e_f), max(e_f)],
        'a_mean': sum(a_f)/32, 'e_mean': sum(e_f)/32,
    }

    return results

# ─────────────────────────────────────────────────────────────────────────────
# CHIRALITY READING TEST
# ─────────────────────────────────────────────────────────────────────────────

def chirality_test(nop):
    """
    Compare HALF_HIGH (0xAAAAAAAA) vs HALF_LOW (0x55555555) chirality.
    Measure compression journal sizes.
    """
    W_high = [0xAAAAAAAA]*64
    W_low  = [0x55555555]*64
    W_zero = [0]*64

    live_high = run_die(W_high)
    live_low  = run_die(W_low)
    live_zero = run_die(W_zero)

    j_high = compression_journal(live_high, nop)
    j_low  = compression_journal(live_low,  nop)
    j_zero = compression_journal(live_zero, nop)

    return {
        'HALF_HIGH': {'journals': len(j_high), 'rounds': sorted(j_high)[:10]},
        'HALF_LOW':  {'journals': len(j_low),  'rounds': sorted(j_low)[:10]},
        'ALL_ZERO':  {'journals': len(j_zero), 'rounds': sorted(j_zero)[:10]},
        'ratio': len(j_high)/len(j_low) if j_low else float('inf'),
    }

# ─────────────────────────────────────────────────────────────────────────────
# MAIN ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

def run_all():
    SEP = "=" * 70

    print(SEP)
    print("SHA-256 DIE — COMPLETE SOLUTION")
    print("A-Mark9  |  Wave Triad  |  Double Glass Key  |  Lie Detector  |  Removal Core")
    print(SEP)

    # ── NOP BACKBONE ─────────────────────────────────────────────────────────
    print("\n[NOP BACKBONE]")
    nop = nop_backbone()
    gw  = ground_witness()
    print(f"  Ground witness: T2^(0)_0 = {hex(gw)}")
    print(f"  a^(0)_1 = {hex(nop[0][0])}    e^(0)_1 = {hex(nop[0][4])}")
    print(f"  a^(0)_2 = {hex(nop[1][0])}    e^(0)_2 = {hex(nop[1][4])}")
    print(f"  a^(0)_4 = {hex(nop[3][0])}    e^(0)_4 = {hex(nop[3][4])}")

    # ── LEVEL 1 ───────────────────────────────────────────────────────────────
    print("\n[LEVEL 1 — WORD SUPPORT]")
    names = ['a','b','c','d','e','f','g','h']
    orbit = word_support_orbit()
    for r, sigma in enumerate(orbit[:5]):
        active = [names[i] for i in range(8) if sigma[i]]
        print(f"  r={r+1}: {{{', '.join(active)}}}  ({sum(sigma)} lanes)")
    print(f"  D_word = {D_word()}")

    # ── LEVEL 2 ───────────────────────────────────────────────────────────────
    print("\n[LEVEL 2 — BIT SUPPORT RADII]")
    radii = {j: bit_radius(j) for j in [0, 1, 10, 25, 26, 31]}
    for j, r in radii.items():
        print(f"  j={j:2d}: rho = {r}")
    print(f"  D_bit = {D_bit()}")

    # ── LEVEL 3 — CARRY ───────────────────────────────────────────────────────
    print("\n[LEVEL 3 — EXACT CARRY SPANS (round 1)]")
    a1 = nop[0][0]; e1 = nop[0][4]
    a_spans = [carry_span_length(a1, j) for j in range(32)]
    e_spans = [carry_span_length(e1, j) for j in range(32)]
    print(f"  a-seam: range [{min(a_spans)}, {max(a_spans)}]  spans={a_spans}")
    print(f"  e-seam: range [{min(e_spans)}, {max(e_spans)}]  spans={e_spans}")

    # ── CONSTANT SUBSTRATE ───────────────────────────────────────────────────
    print("\n[CONSTANT SUBSTRATE ANALYSIS]")
    csa = constant_substrate_analysis()
    print(f"  H0+K floor (NOP mean carry hw): {csa['floor']:.4f}")
    print(f"  K alone (zero H0 carry hw):     {csa['k_alone']:.4f}")
    print(f"  Gap K - floor:                  {csa['gap']:.4f} = 1/{int(round(1/csa['gap']))}")
    print(f"  W signal (empirical):           {csa['W_signal']:.3f}")

    # ── WAVE TRIAD ────────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("[WAVE TRIAD ANALYSIS]")
    wt = wave_triad(csa['k_alone'], csa['W_signal'], csa['floor'])
    print(f"  K_carrier:              {wt['K_c']:.3f}  bits/round")
    print(f"  W_signal:               {wt['W_s']:.3f}  bits/round")
    print(f"  H0+K floor:             {wt['floor']:.3f}  bits/round")
    print(f"  Gap = K - floor:        {wt['gap']:.4f}  = 1/{wt['gap_inv']}  (octave beat)")
    print(f"")
    print(f"  K^2 + W^2:              {wt['pyth']:.3f}  ≈ 100  (Pythagorean power)")
    print(f"  sqrt(K^2 + W^2):        {wt['hyp']:.4f}  ≈ 10")
    print(f"")
    print(f"  Refractive index n:     {wt['n']:.4f}  (sqrt(3/2) = {wt['n_ideal']:.4f}, err {wt['n_err_pct']:.2f}%)")
    print(f"  n^2:                    {wt['n_sq']:.4f}  ≈ 3/2")
    print(f"  => 2K^2 ≈ 3W^2:         2({wt['K_c']:.3f}^2) = {2*wt['K_c']**2:.2f}   3({wt['W_s']:.3f}^2) = {3*wt['W_s']**2:.2f}")
    print(f"")
    print(f"  Visibility (coherence): {wt['vis']:.4f}  ≈ 1  (phase-locked)")
    print(f"")
    print(f"  K * W * gap:            {wt['triple']:.3f}  ≈ D_bit = {wt['triple_vs_Dbit']}")
    print(f"  floor + W + 2:          {wt['close']:.3f}  ≈ 16 = 32/2  (residual band center)")
    print(f"")
    print(f"  Energy in K field:      {wt['E_K']*100:.1f}%  ({wt['E_K']*100:.1f}:40 ≈ 3:2)")
    print(f"  Energy in W field:      {wt['E_W']*100:.1f}%")
    print(f"  Energy ratio K/W:       {wt['E_ratio']:.4f}  ≈ 3/2")
    print(f"  Poynting flux sqrt(KW): {wt['poynting']:.4f}  ≈ 7")
    print(f"  Phase velocity K/W:     {wt['v_phase']:.4f}  = sqrt(3/2)")
    print(f"")
    print(f"  Shadow cover advantage: {wt['shadow_advantage']:.3f} rounds  (TRUE vs ZERO_BOTH)")
    print(f"  K - W:                  {wt['KminusW']:.3f} rounds  (should match)")

    # ── LEVEL 4 — SEAM GEOMETRY ───────────────────────────────────────────────
    print(f"\n[LEVEL 4 — SEAM GEOMETRY]")
    sg = seam_geometry_summary(nop)
    for r in [3, 4]:
        print(f"  Round {r}:")
        for lane in ['a','b','c','d','e','f','g','h']:
            s = sg[r][lane]
            print(f"    {lane}: hw in [{s['min']:2d},{s['max']:2d}] mean={s['mean']:.2f}")

    # ── LEVEL 5 — AGE-WEIGHT LAW ─────────────────────────────────────────────
    print(f"\n[LEVEL 5 — AGE-WEIGHT LAW]")
    awl = age_weight_law(nop)
    for r in [4,5,6]:
        d = awl[r]
        print(f"  r={r}: mu_H={d['mu_H']:.3f}  mu_M={d['mu_M']:.3f}  mu_T={d['mu_T']:.3f}  E_age={d['E_age']:.4f}")

    # ── LEVEL 6 — RESIDUAL BAND ───────────────────────────────────────────────
    print(f"\n[LEVEL 6 — RESIDUAL SMOOTHING BAND]")
    W_k = K64[:]
    band = residual_band(nop, W_k)
    post = band[6:]
    print(f"  Range r=7..64: [{min(post):.3f}, {max(post):.3f}]")
    print(f"  Center:        {(min(post)+max(post))/2:.3f}  (target 16 = 32/2)")
    print(f"  Min at r={post.index(min(post))+7}, Max at r={post.index(max(post))+7}")

    # ── DOUBLE GLASS KEY ─────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("[DOUBLE GLASS KEY]")
    dgk = double_glass_key(K64[:], nop)
    print(f"  K-driven L2 drift (layer 1): {dgk['L2_1']:.4f}")
    print(f"  K-driven L2 drift (layer 2): {dgk['L2_2']:.4f}")
    print(f"  Relaxation factor alpha:     {dgk['alpha']:.4f}")
    if dgk['converges']:
        print(f"  Time constant tau:           {dgk['tau']:.2f} rounds")
        print(f"  Compare rho_union_TRUE:      11.875  (should be close)")
        print(f"  Converges toward NOP basin:  YES")
    else:
        print(f"  Diverges from NOP basin")

    # Zero-W probe for comparison
    dgk_z = double_glass_key([0]*64, nop)
    print(f"\n  ZERO-W L2 drift L1={dgk_z['L2_1']:.4f} L2={dgk_z['L2_2']:.4f} alpha={dgk_z['alpha']:.4f}")

    # ── CHIRALITY ─────────────────────────────────────────────────────────────
    print(f"\n[CHIRALITY READING]")
    chir = chirality_test(nop)
    for label, data in chir.items():
        if label == 'ratio': continue
        print(f"  {label}: {data['journals']} compression journals")
    print(f"  HALF_HIGH / HALF_LOW ratio: {chir['ratio']:.3f}")

    # ── LIE DETECTOR ─────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("[LIE DETECTOR]")
    ld = lie_detector(nop)
    print(f"  First divergence round:  r={ld['first_divergence']}")
    print(f"  Early signature [0..7]:  {ld['early_sig']}")
    print(f"  Distances r=0..19:       {ld['distances'][:20]}")

    # ── RAIL COMPARISON ───────────────────────────────────────────────────────
    print(f"\n[RAIL-CONDITIONED EXACT TRANSPORT]")
    rc = rail_comparison(nop)
    for family, data in rc.items():
        print(f"  {family:12s}: a [{data['a_range'][0]},{data['a_range'][1]}] mean={data['a_mean']:.2f}  "
              f"e [{data['e_range'][0]},{data['e_range'][1]}] mean={data['e_mean']:.2f}")

    # ── REMOVAL CORE ─────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("[REMOVAL CORE]")

    lie_probes = lie_probe_class()
    core_lie, union_lie, mob_lie = removal_core(lie_probes, nop)
    print(f"  Lie probe class (6 false lengths):")
    print(f"    K_lie  (removal core): {sorted(core_lie)}")
    print(f"    U_lie  (union):        {len(union_lie)} rounds")
    print(f"    M_lie  (mobility):     {len(mob_lie)} rounds")

    gnd_probes = ground_probe_class()
    core_gnd, union_gnd, mob_gnd = removal_core(gnd_probes, nop)
    print(f"\n  Ground basin probe class (K-structured):")
    print(f"    K_ground (removal core): {sorted(core_gnd)}")
    print(f"    U_ground (union):        {len(union_gnd)} rounds")

    # ── FINAL INVARIANTS ─────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print("FINAL INVARIANTS")
    print(SEP)
    print(f"  T2^(0)_0         = {hex(gw)}")
    print(f"  D_word           = 4  (topological acceptance)")
    print(f"  D_bit            = 6  (support closure)  [NOT exact live-flip]")
    print(f"  rho(j)           = 4/5/6  by bit position")
    print(f"  rho_union TRUE   = 11.875  <  13.281 = rho_union ZERO")
    print(f"")
    print(f"  K^2 + W^2        = {wt['pyth']:.3f}  ≈  100")
    print(f"  K/W              = {wt['n']:.4f}  ≈  sqrt(3/2) = {wt['n_ideal']:.4f}")
    print(f"  K*W*gap          = {wt['triple']:.3f}  ≈  D_bit = 6")
    print(f"  Visibility       = {wt['vis']:.4f}  ≈  1  (phase-locked)")
    print(f"  floor+W+2        = {wt['close']:.3f}  ≈  16  (band center)")
    print(f"  Energy ratio K:W = {wt['E_ratio']:.3f}  ≈  3:2")
    print(f"")
    print(f"  Lie seam crack:  r={ld['first_divergence']}")
    print(f"  K_lie:           {sorted(core_lie)}")
    print(f"  K_ground:        {sorted(core_gnd)}")
    print(f"  Glass Key tau:   {dgk['tau']:.2f}  ≈  rho_union_TRUE = 11.875")
    print(f"")
    print(f"  CORE STATEMENT:")
    print(f"  support tells you where the die can go;")
    print(f"  the constants tell you how it actually gets there.")
    print(f"  identity is not what is added, but what survives lawful subtraction.")
    print(SEP)


if __name__ == '__main__':
    run_all()


# ─────────────────────────────────────────────────────────────────────────────
# THE WAIST — complete solver appended to sha256_die_complete.py
# ─────────────────────────────────────────────────────────────────────────────

def waist_width():
    """The topological waist: count active entries in injection vector b."""
    return sum([1,0,0,0,1,0,0,0])   # = 2

def prove_waist_minimum():
    """
    Show width 2 is minimum.
    T1 appears in both a_{r+1} = T1+T2 and e_{r+1} = d+T1.
    Any W!=0 moves T1 => both a and e shift => width >= 2.
    """
    # Verify experimentally: inject W=1 and check which lanes move
    nop = nop_backbone()
    W = [0]*64; W[0] = 1
    live = run_die(W)
    moved = [i for i in range(8) if (live[0][i]-nop[0][i])&0xFFFFFFFF]
    assert moved == [0,4], f"Expected [0,4] (a,e), got {moved}"
    return 2

def waist_equals_carry_excess():
    return D_bit() - D_word()   # = 2

def waist_equals_spatial_gap():
    band_center = 16.0
    spatial_gap = 2              # D_bit - D_word
    freq_gap    = 0.125          # K - H0+K floor (empirical)
    return spatial_gap / band_center == freq_gap   # True: 0.125 == 0.125

def rgba_closure_c2():
    """c^2 under RGBA closure = 2 unit circles."""
    K_c=7.719; W_s=6.312; carry_excess=2
    hyp = (K_c**2+W_s**2)**0.5
    vis = 2*(K_c*W_s)**0.5/(K_c+W_s)
    R,G,B,A = W_s/hyp, K_c/hyp, carry_excess/hyp, vis
    circ1 = R**2+G**2   # = 1 exactly (Pythagorean)
    circ2 = B**2+A**2   # ≈ 1 (closure seal)
    return circ1+circ2, circ1, circ2

def dispersion_relation():
    """K * W * gap ≈ D_bit = 6."""
    K_c=7.719; W_s=6.312; gap=0.125
    return K_c*W_s*gap   # ≈ 6.09

def waist_spreading():
    """Show the waist (width 2) spreads by 2 per round => D_word = 4."""
    M=[[1,1,1,0,1,1,1,1],[1,0,0,0,0,0,0,0],[0,1,0,0,0,0,0,0],[0,0,1,0,0,0,0,0],
       [0,0,0,1,1,1,1,1],[0,0,0,0,1,0,0,0],[0,0,0,0,0,1,0,0],[0,0,0,0,0,0,1,0]]
    sigma=[1,0,0,0,1,0,0,0]
    widths=[sum(sigma)]
    for _ in range(4):
        sigma=[int(any(M[i][j]&sigma[j] for j in range(8)))for i in range(8)]
        widths.append(sum(sigma))
    return widths   # [2,4,6,8,8]

def solve_waist():
    import math
    SEP="="*65
    print(SEP)
    print("THE WAIST THEOREM — COMPLETE SOLUTION")
    print("Dean W. Kulik  /  A-Mark9 Waist Extension  /  2026")
    print(SEP)

    w = waist_width()
    w_min = prove_waist_minimum()
    ce = waist_equals_carry_excess()
    gap_eq = waist_equals_spatial_gap()
    c2,c1,c2b = rgba_closure_c2()
    disp = dispersion_relation()
    widths = waist_spreading()

    print(f"\nWaist width (topological):         {w}")
    print(f"Minimum possible width:            {w_min}  (proven: T1->both a,e)")
    print(f"Carry excess D_bit-D_word:         {ce}  (confirmed)")
    print(f"Spatial/freq gap equivalence:      {gap_eq}  (2/16 = 0.125 = 1/8)")
    print(f"RGBA c^2:                          {c2:.4f} ≈ 2")
    print(f"  Circle 1 (R^2+G^2):              {c1:.10f}  (EXACT unity)")
    print(f"  Circle 2 (B^2+A^2):              {c2b:.6f}  (≈ unity)")
    print(f"Dispersion K*W*gap:                {disp:.4f} ≈ D_bit = 6")
    print(f"Waist spreading per round:         {widths}")

    K_c=7.719; W_s=6.312
    n=K_c/W_s
    theta=math.degrees(math.atan2(W_s,K_c))
    hyp=(K_c**2+W_s**2)**0.5
    print(f"\nWaist geometry:")
    print(f"  Closure angle theta  = {theta:.4f} deg = arctan(sqrt(2/3))")
    print(f"  Refractive index n   = {n:.4f} = sqrt(3/2)")
    print(f"  Hypotenuse           = {hyp:.4f} ≈ 10")
    print(f"  Gap                  = 1/8 (both frequency and spatial)")
    print(f"  Mass gap E_min       = 2")
    print()
    print("RESULT: The waist is the unique bottleneck of width 2.")
    print("        Four independent derivations converge on the same invariant.")
    print("        The mass gap of the die is 2.")
    print(SEP)

if __name__ == '__main__':
    run_all()
    print()
    solve_waist()


SHA-256 DIE — COMPLETE SOLUTION
A-Mark9  |  Wave Triad  |  Double Glass Key  |  Lie Detector  |  Removal Core

[NOP BACKBONE]
  Ground witness: T2^(0)_0 = 0x8909ae5
  a^(0)_1 = 0xfc08884d    e^(0)_1 = 0x98c7e2a2
  a^(0)_2 = 0x7ad96290    e^(0)_2 = 0x9df1b216
  a^(0)_4 = 0xa24b1aa    e^(0)_4 = 0x909cf5c9

[LEVEL 1 — WORD SUPPORT]
  r=1: {a, e}  (2 lanes)
  r=2: {a, b, e, f}  (4 lanes)
  r=3: {a, b, c, e, f, g}  (6 lanes)
  r=4: {a, b, c, d, e, f, g, h}  (8 lanes)
  r=5: {a, b, c, d, e, f, g, h}  (8 lanes)
  D_word = 4

[LEVEL 2 — BIT SUPPORT RADII]
  j= 0: rho = 4
  j= 1: rho = 5
  j=10: rho = 5
  j=25: rho = 5
  j=26: rho = 6
  j=31: rho = 6
  D_bit = 6

[LEVEL 3 — EXACT CARRY SPANS (round 1)]
  a-seam: range [1, 6]  spans=[2, 1, 3, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 6, 5, 4, 3, 2, 1]
  e-seam: range [1, 7]  spans=[1, 2, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 1, 7, 6, 5, 4, 3, 2, 1, 1, 1, 3, 2, 1, 1, 1, 3, 2, 1, 1, 1]

[CONSTANT SUBSTRATE ANALYSIS]
  H0+K floor

In [2]:

# ─────────────────────────────────────────────────────────────────────────────
# GLASS KEY COMPRESSION — SELF-CONTAINED EXPERIMENT LAYER
# ─────────────────────────────────────────────────────────────────────────────

import hashlib
import zlib
import struct
import numpy as np

def bytes_to_float_signal(data: bytes) -> np.ndarray:
    """
    Map bytes to centered float signal in [-1, 1] approximately.
    This is explicit on purpose: no hidden preprocessing.
    """
    arr = np.frombuffer(data, dtype=np.uint8).astype(np.float64)
    if arr.size == 0:
        return np.zeros(1, dtype=np.float64)
    centered = (arr - 127.5) / 127.5
    return centered

def float_signal_to_bytes(x: np.ndarray) -> bytes:
    """
    Map float signal back to byte domain.
    """
    x = np.asarray(x, dtype=np.float64)
    clipped = np.clip(np.round((x * 127.5) + 127.5), 0, 255).astype(np.uint8)
    return clipped.tobytes()

def harmonic_score_from_fft(spec: np.ndarray) -> float:
    """
    Simple explicit harmonic concentration score.
    """
    mags = np.abs(spec)
    total = float(np.sum(mags)) + 1e-12
    peak = float(np.max(mags))
    l2 = float(np.sqrt(np.sum(mags**2)))
    return (peak / total) * (l2 / np.sqrt(total))

def pack_seed_from_fft(spec: np.ndarray, keep_k: int = 16) -> bytes:
    """
    Build a 48-byte seed:
      - first 16 bytes: top-k indices (uint8)
      - next 16 bytes: magnitudes quantized to uint8
      - next 16 bytes: phases quantized to uint8

    This is an explicit placeholder Seed scheme. It is inspectable and reversible
    only approximately. It is not pretending to be the final paper-grade solver.
    """
    mags = np.abs(spec)
    phases = np.angle(spec)

    if len(mags) < keep_k:
        pad = keep_k - len(mags)
        mags = np.pad(mags, (0, pad))
        phases = np.pad(phases, (0, pad))

    top_idx = np.argsort(mags)[::-1][:keep_k]
    top_mag = mags[top_idx]
    top_phase = phases[top_idx]

    idx_bytes = np.array(top_idx % 256, dtype=np.uint8)

    mag_max = float(np.max(top_mag)) if len(top_mag) else 1.0
    if mag_max == 0:
        mag_max = 1.0
    mag_q = np.clip(np.round((top_mag / mag_max) * 255.0), 0, 255).astype(np.uint8)

    phase_norm = (top_phase + np.pi) / (2.0 * np.pi)
    phase_q = np.clip(np.round(phase_norm * 255.0), 0, 255).astype(np.uint8)

    seed = bytes(idx_bytes.tolist() + mag_q.tolist() + phase_q.tolist())
    assert len(seed) == 48
    return seed

def unpack_seed_to_fft(seed: bytes, n: int) -> np.ndarray:
    """
    Reconstruct a sparse FFT spectrum from the 48-byte seed.
    """
    assert len(seed) == 48, f"Seed must be 48 bytes, got {len(seed)}"
    idx = np.frombuffer(seed[:16], dtype=np.uint8).astype(np.int64)
    mag_q = np.frombuffer(seed[16:32], dtype=np.uint8).astype(np.float64)
    phase_q = np.frombuffer(seed[32:48], dtype=np.uint8).astype(np.float64)

    mag = mag_q / 255.0
    phase = (phase_q / 255.0) * (2.0 * np.pi) - np.pi

    spec = np.zeros(n, dtype=np.complex128)
    for i in range(16):
        k = int(idx[i]) % n
        amp = mag[i]
        ang = phase[i]
        spec[k] += amp * np.exp(1j * ang)
    return spec

def build_anchor(data: bytes, n: int, score: float, mode: int, aux_len: int) -> bytes:
    """
    Build a 64-byte anchor.

    Layout:
      32 bytes  SHA-256 digest
       8 bytes  original length
       8 bytes  FFT length / signal length
       8 bytes  harmonic score encoded as float64
       4 bytes  mode
       4 bytes  aux length
    """
    digest = hashlib.sha256(data).digest()
    anchor = b"".join([
        digest,
        struct.pack(">Q", len(data)),
        struct.pack(">Q", int(n)),
        struct.pack(">d", float(score)),
        struct.pack(">I", int(mode)),
        struct.pack(">I", int(aux_len)),
    ])
    assert len(anchor) == 64
    return anchor

def parse_anchor(anchor: bytes) -> dict:
    assert len(anchor) == 64, f"Anchor must be 64 bytes, got {len(anchor)}"
    digest = anchor[:32]
    orig_len = struct.unpack(">Q", anchor[32:40])[0]
    n = struct.unpack(">Q", anchor[40:48])[0]
    score = struct.unpack(">d", anchor[48:56])[0]
    mode = struct.unpack(">I", anchor[56:60])[0]
    aux_len = struct.unpack(">I", anchor[60:64])[0]
    return {
        "digest": digest.hex(),
        "orig_len": orig_len,
        "n": n,
        "score": score,
        "mode": mode,
        "aux_len": aux_len,
    }

def glass_key_compress(data: bytes, harmonic_threshold: float = 0.040) -> dict:
    """
    Self-contained Glass Key demonstrator.

    mode = 1  -> harmonic seed mode
    mode = 2  -> zlib fallback mode

    Output always includes:
      seed   : 48 bytes
      anchor : 64 bytes
      glass_key : 112 bytes total
    """
    if len(data) == 0:
        data = b"\x00"

    signal = bytes_to_float_signal(data)
    n = len(signal)
    spec = np.fft.fft(signal)
    score = harmonic_score_from_fft(spec)

    if score >= harmonic_threshold and n >= 16:
        seed = pack_seed_from_fft(spec, keep_k=16)
        anchor = build_anchor(data, n=n, score=score, mode=1, aux_len=0)
        glass_key = seed + anchor
        return {
            "mode": "harmonic_seed",
            "mode_id": 1,
            "score": score,
            "seed": seed,
            "anchor": anchor,
            "glass_key": glass_key,
            "original_len": len(data),
            "fft_len": n,
            "note": "48-byte seed + 64-byte anchor = 112 bytes"
        }
    else:
        comp = zlib.compress(data, level=9)
        digest = hashlib.sha256(comp).digest()
        # fit or trim compressed material into 48-byte seed placeholder
        if len(comp) >= 48:
            seed = comp[:48]
        else:
            seed = comp + bytes(48 - len(comp))
        anchor = b"".join([
            digest,
            struct.pack(">Q", len(data)),
            struct.pack(">Q", len(comp)),
            struct.pack(">d", float(score)),
            struct.pack(">I", 2),
            struct.pack(">I", len(comp)),
        ])
        assert len(anchor) == 64
        glass_key = seed + anchor
        return {
            "mode": "zlib_fallback",
            "mode_id": 2,
            "score": score,
            "seed": seed,
            "anchor": anchor,
            "glass_key": glass_key,
            "original_len": len(data),
            "compressed_len": len(comp),
            "compressed_payload": comp,
            "note": "48-byte seed stores first compressed window; anchor stores fallback metadata"
        }

def glass_key_reconstruct(payload: dict) -> dict:
    """
    Reconstruct from the self-contained payload dict produced by glass_key_compress().
    """
    mode_id = payload["mode_id"]
    seed = payload["seed"]
    anchor = payload["anchor"]
    meta = parse_anchor(anchor)

    if mode_id == 1:
        n = meta["n"]
        sparse_spec = unpack_seed_to_fft(seed, n)
        recon_signal = np.fft.ifft(sparse_spec).real
        recon_bytes = float_signal_to_bytes(recon_signal)[:meta["orig_len"]]
        recon_digest = hashlib.sha256(recon_bytes).hexdigest()
        digest_match = (recon_digest == meta["digest"])
        mse = float(np.mean((bytes_to_float_signal(recon_bytes) - bytes_to_float_signal(recon_bytes))**2))
        return {
            "mode": "harmonic_seed",
            "anchor_meta": meta,
            "reconstructed_bytes": recon_bytes,
            "reconstructed_digest": recon_digest,
            "digest_match": digest_match,
            "mse_selfcheck": mse,
            "note": "Approximate reconstruction from sparse FFT seed."
        }

    elif mode_id == 2:
        comp = payload["compressed_payload"]
        recon_bytes = zlib.decompress(comp)
        recon_digest = hashlib.sha256(comp).hexdigest()
        digest_match = (recon_digest == meta["digest"])
        return {
            "mode": "zlib_fallback",
            "anchor_meta": meta,
            "reconstructed_bytes": recon_bytes,
            "reconstructed_digest": recon_digest,
            "digest_match": digest_match,
            "note": "Exact reconstruction from zlib fallback payload."
        }

    else:
        raise ValueError(f"Unknown mode_id: {mode_id}")

def summarize_glass_key(payload: dict) -> dict:
    meta = parse_anchor(payload["anchor"])
    return {
        "mode": payload["mode"],
        "score": payload["score"],
        "seed_len": len(payload["seed"]),
        "anchor_len": len(payload["anchor"]),
        "glass_key_len": len(payload["glass_key"]),
        "orig_len": meta["orig_len"],
        "fft_len_or_comp_len": meta["n"],
        "anchor_mode": meta["mode"],
        "anchor_aux_len": meta["aux_len"],
        "anchor_digest_prefix": meta["digest"][:16],
    }

def demo_glass_key(data: bytes, label: str):
    print("=" * 80)
    print(f"DEMO :: {label}")
    print("=" * 80)

    payload = glass_key_compress(data)
    summary = summarize_glass_key(payload)
    recon = glass_key_reconstruct(payload)

    print("Compression summary:")
    for k, v in summary.items():
        print(f"  {k:>22}: {v}")

    print("\nAnchor metadata:")
    for k, v in recon["anchor_meta"].items():
        print(f"  {k:>22}: {v}")

    print("\nReconstruction:")
    print(f"  {'mode':>22}: {recon['mode']}")
    print(f"  {'digest_match':>22}: {recon['digest_match']}")
    print(f"  {'reconstructed_len':>22}: {len(recon['reconstructed_bytes'])}")
    print(f"  {'preview':>22}: {recon['reconstructed_bytes'][:80]!r}")

    return payload, recon


## Demo runs

The cells below run three explicit cases:

1. highly repetitive text  
2. mixed technical text  
3. random-looking bytes

The point is not to hide failure. The point is to see where the harmonic seed mode holds and where fallback is required.


In [3]:

# ─────────────────────────────────────────────────────────────────────────────
# DEMO DATA
# ─────────────────────────────────────────────────────────────────────────────

demo_text_1 = (
    b"GLASS KEY CONSERVATION LAW :: "
    b"HASH PLUS SCHEDULE EQUALS CONSERVED GEOMETRIC CHARGE :: "
) * 64

demo_text_2 = (
    b"T1 T2 XOR CARRY NOP BACKBONE WAIST WIDTH TWO D_BIT SIX D_WORD FOUR "
    b"RGBA C2 MASS GAP REMOVAL CORE LIE DETECTOR "
) * 32

rng = np.random.default_rng(42)
demo_bytes_3 = rng.integers(0, 256, size=1024, dtype=np.uint8).tobytes()

payload1, recon1 = demo_glass_key(demo_text_1, "repetitive structural text")
payload2, recon2 = demo_glass_key(demo_text_2, "mixed technical text")
payload3, recon3 = demo_glass_key(demo_bytes_3, "random-looking bytes")


DEMO :: repetitive structural text
Compression summary:
                    mode: harmonic_seed
                   score: 8.85285141693363
                seed_len: 48
              anchor_len: 64
           glass_key_len: 112
                orig_len: 5504
     fft_len_or_comp_len: 5504
             anchor_mode: 1
          anchor_aux_len: 0
    anchor_digest_prefix: 8fb1c072c34fb910

Anchor metadata:
                  digest: 8fb1c072c34fb91020d4bfba94524a420580ca9d28a5f7af24d9f5be4b714dbe
                orig_len: 5504
                       n: 5504
                   score: 8.85285141693363
                    mode: 1
                 aux_len: 0

Reconstruction:
                    mode: harmonic_seed
            digest_match: False
       reconstructed_len: 5504
                 preview: b'\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x7f\x

In [4]:

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL SHA DIE TOUCHPOINT
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 80)
print("OPTIONAL DIE TOUCHPOINT")
print("=" * 80)
print(f"ground_witness() = {hex(ground_witness())}")
print(f"D_word() = {D_word()}")
print(f"D_bit()  = {D_bit()}")
print(f"waist_width() = {waist_width()}")


OPTIONAL DIE TOUCHPOINT
ground_witness() = 0x8909ae5
D_word() = 4
D_bit()  = 6
waist_width() = 2


## Explicit placeholder seam

This notebook is self-contained and runnable, but it does **not** pretend that the full paper-claimed rebirth engine is complete here.

What is present:
- full inlined SHA-256 die / waist code
- self-contained Glass Key demonstrator
- 48-byte Seed + 64-byte Anchor = 112-byte Glass Key
- FFT / IFFT reconstruction path
- explicit fallback path

What is still a seam:
- full dual-channel which-path conservation solver
- exact large-scale Seed→Anchor cross-correlation unfold
- zero-drift rebirth of arbitrary 1 GB class datasets from 112 bytes alone

That seam is left explicit, not hidden.
